# ConvTran Reproduction: UEA Baseline Training

We evaluate the released ConvTran implementation on the 30 datasets in the UEA multivariate archive. The BasicMotions pilot is documented in `01_reproduction.ipynb`; archive preparation and dataset dimensions are documented in `02_uea_benchmark.ipynb`. This notebook records the baseline training protocol, completed results, and limitations encountered as the benchmark progresses.

- Paper: https://doi.org/10.1007/s10618-023-00948-2
- Official code: https://github.com/Navidfoumani/ConvTran
- Project repository: https://github.com/tyrerodr/convtran-reproduction

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)
assert ROOT is not None, "Open this notebook from the project repository."

DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
RUNNER = ROOT / "scripts" / "run_baseline.py"
RUNS_PATH = ROOT / "results" / "baseline_runs.csv"
DIMENSIONS_PATH = ROOT / "results" / "dataset_dimensions.csv"

assert RUNNER.is_file(), f"Missing baseline runner: {RUNNER}"
print("Repository:", ROOT)
print("Python:", sys.executable)
print("Upstream commit:", subprocess.check_output(
    ["git", "-C", str(ROOT / "upstream" / "ConvTran"), "rev-parse", "HEAD"],
    text=True,
).strip())

## Baseline protocol

We keep the released architecture and default training configuration used in the BasicMotions pilot. We call this the *released-code baseline* because the paper's method description differs from some implementation defaults.

| Setting | Baseline value |
|---|---|
| Positional encoding | tAPE and eRPE |
| Embedding dimension | 16 |
| Attention heads | 8 |
| Feedforward dimension | 256 |
| Optimizer | RAdam |
| Learning rate | 0.001 |
| Dropout | 0.01 |
| Epochs | 100 |
| Batch size | 16 |
| Normalization | Disabled |
| Training seeds | 1234, 2345, 3456, 4567, 5678 |
| Model selection | Lowest validation loss |

Each seed runs in a separate Python process. The upstream loader creates one stratified validation split with a fixed random state of 1234, independent of the training seed. Thus the five runs measure variation in training, not variation across data splits. Test accuracy is computed on the official UEA test split after checkpoint selection.

## Dataset scope

The dataset inventory contains 30 TRAIN/TEST pairs. It records sample counts, channel counts, class counts, and sequence lengths without loading all series into memory. File presence and the inventory do not guarantee that every dataset can be trained with the available GPU memory. In particular, EigenWorms has 17,984 time steps and is scheduled last; its feasibility with the unchanged baseline remains unverified.

In [ ]:
UEA_DATASETS = [
    "ArticularyWordRecognition", "AtrialFibrillation", "BasicMotions",
    "CharacterTrajectories", "Cricket", "DuckDuckGeese", "EigenWorms",
    "Epilepsy", "ERing", "EthanolConcentration", "FaceDetection",
    "FingerMovements", "HandMovementDirection", "Handwriting", "Heartbeat",
    "InsectWingbeat", "JapaneseVowels", "Libras", "LSST", "MotorImagery",
    "NATOPS", "PenDigits", "PEMS-SF", "PhonemeSpectra", "RacketSports",
    "SelfRegulationSCP1", "SelfRegulationSCP2", "SpokenArabicDigits",
    "StandWalkJump", "UWaveGestureLibrary",
]
EXPECTED_SEEDS = {1234, 2345, 3456, 4567, 5678}
assert len(UEA_DATASETS) == len(set(UEA_DATASETS)) == 30

available = {
    name for name in UEA_DATASETS
    if all(
        (DATA_ROOT / name / f"{name}_{split}.ts").is_file()
        for split in ("TRAIN", "TEST")
    )
}
print(f"Datasets with both split files: {len(available)}/30")
if missing := sorted(set(UEA_DATASETS) - available):
    raise FileNotFoundError(f"Missing split files: {missing}")

if DIMENSIONS_PATH.is_file():
    dimensions = pd.read_csv(DIMENSIONS_PATH)
    display(dimensions[
        ["dataset", "train_samples", "test_samples",
         "channels", "classes", "max_length", "variable_length"]
    ])
else:
    print("Dataset dimensions CSV not found; see notebook 02.")

## Training

The runner invokes the official `main.py` separately for each seed and writes logs and checkpoints under `outputs/baseline/<dataset>/seed<seed>/`. It reuses a completed run when its log contains the final test summary. It writes a dataset's rows to `results/baseline_runs.csv` after all five seeds have completed. An incomplete log must be inspected before that seed is retried.

The cell below trains datasets that are not yet complete in the CSV. It runs one dataset at a time; if one fails, the failure is reported and the next dataset is attempted. No model or batch-size change is made automatically. Running this cell starts training and may take substantial time.

In [ ]:
def load_runs():
    if not RUNS_PATH.is_file():
        return pd.DataFrame(columns=[
            "dataset", "seed", "epochs", "batch_size", "accuracy", "test_loss"
        ])
    runs = pd.read_csv(RUNS_PATH)
    expected = [
        "dataset", "seed", "epochs", "batch_size", "accuracy", "test_loss"
    ]
    if list(runs.columns) != expected:
        raise ValueError(f"Unexpected results columns: {list(runs.columns)}")
    if runs.duplicated(["dataset", "seed"]).any():
        raise ValueError("Duplicate dataset/seed rows in the results CSV")
    return runs

def completed_datasets(runs):
    return {
        dataset for dataset, group in runs.groupby("dataset")
        if len(group) == 5
        and set(group["seed"].astype(int)) == EXPECTED_SEEDS
    }

runs = load_runs()
print("Completed:", sorted(completed_datasets(runs)))
print("Pending:", sorted(set(UEA_DATASETS) - completed_datasets(runs)))

In [ ]:
training_order = [
    name for name in UEA_DATASETS if name != "EigenWorms"
] + ["EigenWorms"]

attempts = []
for dataset in training_order:
    if dataset in completed_datasets(load_runs()):
        continue

    print(f"\nTraining {dataset}", flush=True)
    process = subprocess.run(
        [sys.executable, str(RUNNER), "--dataset", dataset],
        cwd=ROOT,
        check=False,
    )
    attempts.append({
        "dataset": dataset,
        "exit_code": process.returncode,
    })
    if process.returncode:
        print(
            f"{dataset} failed. Inspect "
            f"{ROOT / 'outputs' / 'baseline' / dataset.lower()}",
            flush=True,
        )

display(pd.DataFrame(attempts, columns=["dataset", "exit_code"]))

## Results collected so far

Before the remaining benchmark, BasicMotions and Epilepsy each have five runs at 100 epochs and batch size 16. BasicMotions reached 100% test accuracy in all five. Epilepsy reached 96.38% in all five, corresponding to 133 of 138 test samples classified correctly per run. These figures come from the recorded results CSV; identical accuracies do not prove identical predictions.

The table below is generated from the CSV after training. It reports a mean and sample standard deviation only for datasets with all five expected seeds. Partial or failed experiments remain visible in the progress table.

In [ ]:
runs = load_runs()
completed = completed_datasets(runs)

if runs.empty:
    print("No baseline results have been recorded.")
else:
    summary = (
        runs.assign(accuracy_pct=runs["accuracy"] * 100)
        .groupby("dataset")
        .agg(
            runs=("seed", "count"),
            mean_accuracy_pct=("accuracy_pct", "mean"),
            std_accuracy_pp=("accuracy_pct", "std"),
        )
        .reset_index()
    )
    summary = summary[summary["dataset"].isin(completed)]
    display(summary.round(4).sort_values("dataset"))

progress = pd.DataFrame({"dataset": UEA_DATASETS})
progress["complete"] = progress["dataset"].isin(completed)
print(f"Completed datasets: {len(completed)}/30")
display(progress)

## Reproducibility findings and limitations

The released code differs from the paper's description in embedding dimension (16 by default versus 64 in the paper), optimizer (RAdam versus Adam), feedforward activation (ReLU versus GELU), and training termination (fixed epoch budget with a best validation-loss checkpoint versus described early stopping). Changing only the embedding dimension also changes the temporal convolution filter count. We therefore keep these results labeled as a released-code baseline.

The processed dataset cache is not automatically invalidated when preprocessing options change. The upstream seed handling also does not establish full determinism, and the five runs share one validation split. These details must be considered when interpreting variability or testing an improvement. The current CSV does not contain resource use or failure causes; those should be recorded from logs if encountered.

## Comparison with the published results

A numerical comparison with the paper is pending verification of its dataset values and aggregation protocol. The final table will distinguish published accuracy from our five-seed mean and will document datasets that could not be completed with the unchanged baseline. A failed run will not be replaced silently with a changed batch size or architecture.

The next notebook will define and evaluate an experimental improvement after the baseline results and discrepancies have been reviewed.